In [ ]:
import numpy as np
from datasets import load_dataset
from  transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

LOAD & SUBSET THE DATASET

In [ ]:
raw_dataset = load_dataset("stanfordnlp/imdb")

train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(5000))
eval_dataset = raw_dataset["test"].shuffle(seed=42).select(range(1000))

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")
# Label distribution check — should be roughly 50/50
print(f"Train label counts: ", train_dataset.to_pandas()["label"].value_counts().to_dict())


TOKENIZE

In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_fn(batch):
  return tokenizer(
      batch["text"],
      truncation=True,
      max_length=128
  )
  # DO NOT pad here — DataCollatorWithPadding does dynamic padding per batch
  # padding here would pad ALL sequences to max_length=128 globally (wastes memory)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_eval = eval_dataset.map(tokenize_fn, batched=True)

#Remove the raw text column — model only needs input_ids, attention_mask, label
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval = tokenized_eval.rename_column("label", "labels")

# Set format to PyTorch tensors
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

# Dynamic padding: pads each batch to the longest sequence in THAT batch (not globally)
# Saves memory vs. padding everything to 128
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

LOAD MODEL

In [ ]:
# num_labels=2: adds a 2-class linear head on top of [CLS] token output
# You'll see a warning about "newly initialized weights" — that's the classification head
# It's expected: the head didn't exist in the pre-trained checkpoint
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)


DEFINE compute_metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  # Logits shape: (batch_size, num_labels) — take argmax for predicted class
  predictions = np.argmax(logits, axis=-1)
  return accuracy_metric.compute(predictions=predictions, references=labels)

TRAINING ARGUMENTS

In [ ]:
training_args = TrainingArguments(
    output_dir = "./bert-imbd-checkpoints",     # saves checkpoints here
    num_train_epochs = 3,
    # Batch sizes
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    learning_rate=2e-5,
    weight_decay=0.01,                      # mild L2 regularisation

    # Evaluation + logging
    eval_strategy="epoch",            # eval after every epoch
    logging_strategy="steps",
    logging_steps=50,                       # log loss every 50 steps

    # Saving
    save_strategy="epoch",
    load_best_model_at_end=True,            # auto-loads the best checkpoint at end
    metric_for_best_model="accuracy",

    # Reproducibility
    seed=42,

    # Mixed precision: fp16 halves VRAM usage on CUDA — critical for GTX 1650
    fp16=True,
)

CREATE TRAINER + RUN

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting fine-tuning...\n")
trainer.train()

EVALUATE + SAVE

In [ ]:
eval_results = trainer.evaluate()
print(f"\n✅ Final Eval Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"   Final Eval Loss:     {eval_results['eval_loss']:.4f}")

# Save the final model + tokenizer
trainer.save_model("./bert-imdb-final")
tokenizer.save_pretrained("./bert-imdb-final")